# Notebook T4.4 — Agente de Análisis de Datos con Herramientas
## Patrón: "LLM que orquesta funciones ML como herramientas"

**Referencia en la guía metodológica:** PASO 5 → PATRÓN D  
**Nivel:** Intermedio-Avanzado

---

### ¿Qué es un Agente y en qué se diferencia de una Chain?

Hasta ahora hemos visto **Chains**: flujos fijos donde los pasos están predefinidos.  
Una Chain siempre ejecuta: paso 1 → paso 2 → paso 3.

Un **Agente** es diferente: el LLM **decide en tiempo real** qué herramientas usar 
y en qué orden, basándose en la pregunta del usuario. Es como la diferencia entre 
un formulario (chain) y una conversación con un analista experto (agente).

```
Chain fija:    pregunta → paso_A → paso_B → paso_C → respuesta
Agente:        pregunta → LLM decide → tool_X() → LLM decide → tool_Y() → respuesta
```

**Caso de uso:** Interfaz conversacional sobre un dataset de ventas.  
El analista hace preguntas en lenguaje natural y el agente usa funciones ML/estadísticas 
para responder con datos reales.

---

### Conceptos nuevos en este notebook

| Concepto | Qué es |
|----------|--------|
| `@tool` | Decorador que convierte una función Python en herramienta del agente |
| `AgentExecutor` | El "motor" que ejecuta el loop: pregunta → tool → resultado → respuesta |
| `MessagesPlaceholder` | Espacio en el prompt para el historial de conversación |
| `create_tool_calling_agent` | Crea un agente que puede llamar funciones externas |
| `verbose=True` | Muestra el "razonamiento" del agente (qué decide hacer y por qué) |

---

### Conexión con la guía metodológica

- **Paso 1:** Interfaz conversacional sobre datos — TIPO C de datos  
- **Paso 3:** El system prompt define la personalidad y capacidades del agente  
- **Paso 5:** PATRÓN D — agente LLM con herramientas ML/estadísticas  
- **Paso 8:** Las herramientas son funciones Python reutilizables con `@tool`

## PASO 2 — Instalación y Configuración

In [ ]:
!pip install langchain langchain-google-genai scikit-learn pandas numpy python-dotenv -q

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# Los agentes necesitan un modelo que soporte "tool calling" (function calling)
# Gemini Flash y Claude Sonnet lo soportan perfectamente
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0  # el agente debe razonar de forma determinista
)
print("✅ LLM con capacidad de tool calling configurado")

## PASO 4 — Dataset de Ventas Global

### Variable global DATASET: por qué es necesaria

Las herramientas del agente son funciones Python que se ejecutan de forma autónoma.  
Para que accedan al dataset sin tener que pasarlo como argumento en cada llamada,  
lo almacenamos como variable global `DATASET`.

En producción, esto sería una conexión a base de datos o un dataframe cargado en memoria.

In [ ]:
np.random.seed(42)
n = 200

DATASET = pd.DataFrame({
    "producto": np.random.choice(["Laptop", "Tablet", "Auriculares", "Teclado", "Monitor"], n),
    "region": np.random.choice(["Norte", "Sur", "Este", "Oeste"], n),
    "mes": np.random.randint(1, 13, n),
    "ventas_unidades": np.random.randint(10, 500, n),
    "precio_unitario": np.random.uniform(20, 800, n).round(2),
    "satisfaccion_cliente": np.random.uniform(2.0, 5.0, n).round(1),
    "descuento_pct": np.random.uniform(0, 0.3, n).round(2),
})

DATASET["revenue"] = (
    DATASET["ventas_unidades"] * DATASET["precio_unitario"] * (1 - DATASET["descuento_pct"])
).round(2)

print(f"Dataset de ventas: {DATASET.shape[0]} registros, {DATASET.shape[1]} columnas")
print(f"\nColumnas disponibles: {list(DATASET.columns)}")
DATASET.head()

## PASO 5 — Definir las Herramientas del Agente

### El decorador `@tool`: cómo convertir una función en herramienta

El decorador `@tool` de LangChain hace tres cosas automáticamente:
1. **Registra la función** como herramienta disponible para el agente
2. **Usa el docstring** como descripción para que el LLM sepa cuándo usarla
3. **Usa los type hints** para que el LLM sepa qué parámetros pasar

**La descripción del docstring es crítica**: el LLM la lee para decidir cuál herramienta 
usar. Una mala descripción hace que el agente elija mal. Sé específico y menciona 
exactamente qué hace la función y con qué tipo de preguntas responde.

### Las 5 herramientas del agente

| Herramienta | Qué hace | Qué técnica ML usa |
|-------------|----------|---------------------|
| `obtener_estadisticas` | Stats descriptivas de una columna | pandas describe() |
| `top_productos_por_revenue` | Ranking de productos | groupby + sort |
| `segmentar_clientes_kmeans` | Perfiles de segmentos | K-Means clustering |
| `predecir_revenue` | Revenue esperado para un escenario | Regresión lineal |
| `comparar_regiones` | Comparativa regional | groupby + agg |

In [ ]:
@tool
def obtener_estadisticas(columna: str) -> str:
    """
    Obtiene estadísticas descriptivas de una columna del dataset de ventas.
    Úsala cuando el usuario pregunte por promedio, máximo, mínimo, distribución.
    Columnas disponibles: producto, region, mes, ventas_unidades, precio_unitario,
    satisfaccion_cliente, descuento_pct, revenue
    """
    if columna not in DATASET.columns:
        return f"Error: columna '{columna}' no existe. Disponibles: {list(DATASET.columns)}"
    if DATASET[columna].dtype == "object":
        # Para columnas categóricas devuelve la distribución de frecuencias
        stats = DATASET[columna].value_counts().to_dict()
        return f"Distribución de '{columna}': {json.dumps(stats, ensure_ascii=False)}"
    else:
        # Para columnas numéricas devuelve count, mean, std, min, max, percentiles
        stats = DATASET[columna].describe().round(2).to_dict()
        return f"Estadísticas de '{columna}': {json.dumps(stats)}"


@tool
def top_productos_por_revenue(n: int = 5) -> str:
    """
    Retorna los N productos con mayor revenue total en el dataset.
    Úsala cuando el usuario pregunte cuál producto vende más o genera más ingresos.
    Parámetro n: número de productos a mostrar (default 5).
    """
    top = (DATASET.groupby("producto")["revenue"]
           .sum().sort_values(ascending=False).head(n).round(2).to_dict())
    return f"Top {n} productos por revenue total: {json.dumps(top)}"


@tool
def segmentar_clientes_kmeans(n_clusters: int = 3) -> str:
    """
    Aplica K-Means clustering sobre precio, ventas, satisfacción y descuento.
    Úsala cuando el usuario pida segmentar, agrupar o encontrar perfiles de clientes/ventas.
    n_clusters debe ser entre 2 y 5.
    """
    features = ["precio_unitario", "ventas_unidades", "satisfaccion_cliente", "descuento_pct"]
    X = DATASET[features].copy()
    # StandardScaler: normaliza las features para que K-Means no esté dominado por la escala
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    DATASET["segmento"] = km.fit_predict(X_scaled)

    perfil = DATASET.groupby("segmento")[features + ["revenue"]].mean().round(2)
    sizes = DATASET["segmento"].value_counts().to_dict()

    result = {}
    for seg in range(n_clusters):
        result[f"Segmento_{seg}"] = {**perfil.loc[seg].to_dict(), "tamaño": sizes[seg]}
    return f"Segmentación K-Means ({n_clusters} clusters): {json.dumps(result)}"


@tool
def predecir_revenue(precio: float, unidades: int, descuento: float) -> str:
    """
    Predice el revenue esperado para un escenario hipotético usando regresión lineal.
    Úsala cuando el usuario haga preguntas del tipo 'si vendo X unidades a Y precio, ¿cuánto gano?'
    precio: precio unitario; unidades: cantidad a vender; descuento: entre 0 y 1 (ej: 0.15 = 15%)
    """
    X = DATASET[["precio_unitario", "ventas_unidades", "descuento_pct"]]
    y = DATASET["revenue"]
    modelo = LinearRegression()
    modelo.fit(X, y)
    r2 = r2_score(y, modelo.predict(X))
    pred = modelo.predict([[precio, unidades, descuento]])[0]
    return (f"Revenue predicho: ${pred:,.2f} (R²={r2:.3f}) "
            f"para precio=${precio}, unidades={unidades}, descuento={descuento:.0%}")


@tool
def comparar_regiones(metrica: str = "revenue") -> str:
    """
    Compara el rendimiento de las 4 regiones (Norte, Sur, Este, Oeste) para una métrica.
    Úsala cuando el usuario quiera saber qué región es mejor, peor, o compararlas entre sí.
    Métricas válidas: revenue, ventas_unidades, satisfaccion_cliente, descuento_pct
    """
    if metrica not in ["revenue", "ventas_unidades", "satisfaccion_cliente", "descuento_pct"]:
        return f"Métrica '{metrica}' no válida."
    comparacion = (DATASET.groupby("region")[metrica]
                   .agg(["mean", "sum", "count"]).round(2).to_dict())
    return f"Comparación por región ({metrica}): {json.dumps(comparacion)}"


tools = [obtener_estadisticas, top_productos_por_revenue,
         segmentar_clientes_kmeans, predecir_revenue, comparar_regiones]

print(f"✅ {len(tools)} herramientas registradas para el agente:")
for t in tools:
    print(f"  • {t.name}: {t.description[:70]}...")

## PASO 5 (continuación) — Construir el Agente

### El System Prompt del Agente

El system prompt del agente es más complejo que el de una chain porque:
1. Define la **personalidad** del agente ("DataBot, analista experto")
2. Describe **qué datos tiene disponibles** para responder
3. Da instrucciones de **comportamiento**: cómo debe razonar, en qué idioma responder

### MessagesPlaceholder: memoria de conversación

`MessagesPlaceholder(variable_name="chat_history")` reserva un espacio en el prompt 
para el historial de conversación. El agente puede ver las preguntas anteriores 
y dar respuestas coherentes con el contexto previo.

### AgentExecutor: el loop de ejecución

`AgentExecutor` implementa el loop **ReAct** (Reason + Act):
1. El LLM recibe la pregunta y **razona** qué herramienta usar
2. Llama a la herramienta y recibe el resultado
3. Si necesita más información, llama a otra herramienta
4. Cuando tiene suficiente información, genera la respuesta final
5. `max_iterations=5` evita loops infinitos

In [ ]:
system_prompt = """
Eres DataBot, un analista de datos experto y amigable. Tienes acceso a herramientas para analizar
un dataset de ventas con productos (Laptop, Tablet, Auriculares, Teclado, Monitor),
regiones (Norte, Sur, Este, Oeste) y métricas: ventas_unidades, precio_unitario, revenue,
satisfaccion_cliente y descuento_pct.

Cuando el usuario haga preguntas:
1. Usa las herramientas apropiadas para obtener datos reales — NUNCA inventes números
2. Si la pregunta requiere múltiples herramientas, úsalas en secuencia
3. Interpreta los resultados con contexto de negocio (no solo datos crudos)
4. Da recomendaciones accionables basadas en los datos
5. Responde siempre en español con un tono profesional pero accesible
"""

# El prompt del agente DEBE tener agent_scratchpad — es donde el agente escribe su razonamiento
prompt_agente = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder(variable_name="chat_history", optional=True),  # historial conversación
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),  # área de trabajo del agente
])

# create_tool_calling_agent combina el LLM con las herramientas
agente = create_tool_calling_agent(llm, tools, prompt_agente)

# AgentExecutor es el "runner" que ejecuta el loop hasta obtener la respuesta final
ejecutor = AgentExecutor(
    agent=agente,
    tools=tools,
    verbose=True,       # muestra el razonamiento y las llamadas a herramientas
    max_iterations=5    # máximo de tool calls por pregunta (evita loops)
)

print("✅ Agente DataBot configurado y listo para responder preguntas")

## PASO 6 — Interacción con el Agente: Preguntas de Ejemplo

### Qué observar en la salida verbose

Cuando `verbose=True`, verás el **razonamiento interno del agente**:
- `> Entering new AgentExecutor chain...` — el agente empieza a trabajar
- `Action: nombre_herramienta` — qué herramienta decide usar
- `Action Input: {parámetros}` — qué le pasa a la herramienta
- `Observation: resultado` — qué devuelve la herramienta
- `Final Answer:` — la respuesta sintetizada

Esto te permite entender exactamente cómo el LLM razona y qué datos usa para responder.

### Las 4 preguntas de prueba progresan en complejidad

1. **Simple**: estadística de una columna → una herramienta
2. **Compuesta**: comparar dos métricas → dos herramientas en secuencia
3. **ML avanzado**: clustering → una herramienta con algoritmo interno
4. **Hipotética**: predicción de escenario → regresión lineal

In [ ]:
def preguntar(pregunta: str):
    """Envía una pregunta al agente y muestra la respuesta formateada."""
    print(f"\n{'='*65}")
    print(f"🧑 PREGUNTA: {pregunta}")
    print(f"{'='*65}")
    result = ejecutor.invoke({"input": pregunta})
    print(f"\n🤖 DATABOT RESPONDE:\n{result['output']}")
    return result


# Pregunta 1: análisis simple — una sola herramienta necesaria
print("--- PREGUNTA 1: Análisis básico ---")
preguntar("¿Cuál es el revenue promedio y qué producto vende más?")

In [ ]:
# Pregunta 2: requiere múltiples herramientas en secuencia
print("--- PREGUNTA 2: Análisis multi-herramienta ---")
preguntar("Compara las regiones por revenue y dime cuál tiene mejor satisfacción del cliente")

In [ ]:
# Pregunta 3: análisis avanzado con Machine Learning (K-Means)
print("--- PREGUNTA 3: Segmentación con ML ---")
preguntar("Segmenta las ventas en 3 grupos y explícame el perfil de cada segmento")

In [ ]:
# Pregunta 4: predicción con regresión lineal
print("--- PREGUNTA 4: Predicción hipotética ---")
preguntar("Si vendo Laptops a $600 con un 10% de descuento y espero vender 100 unidades, ¿cuánto revenue generaría?")

## Conclusiones del Notebook T4.4

### Lo que has aprendido

1. **Agentes vs Chains**: un agente **decide dinámicamente** qué herramientas usar según la pregunta.
   Una chain tiene un flujo fijo. Usa agentes cuando la pregunta puede requerir distintas rutas.

2. **El decorador `@tool`** convierte cualquier función Python en una herramienta que el LLM 
   puede llamar. La clave es escribir un buen docstring — es lo que el LLM lee para decidir.

3. **Tool calling / Function calling**: la capacidad del LLM de llamar funciones externas 
   es la base de los agentes modernos. Claude y Gemini Flash la soportan nativamente.

4. **AgentExecutor con verbose=True** es tu mejor herramienta de debugging: 
   puedes ver exactamente qué decisiones toma el agente y con qué datos.

5. **StandardScaler en K-Means**: siempre normaliza antes de clustering para que 
   las variables con distinta escala no dominen el algoritmo.

---

### ¿Cuándo usar este patrón en proyectos reales?

- Chatbots internos de análisis de datos para equipos de negocio
- Interfaces de consulta sobre dashboards o bases de datos
- Asistentes de BI (Business Intelligence) conversacionales
- Cualquier caso donde distintas preguntas requieran distintos análisis

---

### Limitaciones del patrón de agentes

- Más lento que chains fijas (múltiples llamadas al LLM)
- Puede "alucinarse" si las descripciones de herramientas no son precisas
- El `max_iterations` puede cortar respuestas en preguntas muy complejas
- Más difícil de probar y depurar que una chain predecible

---

### Patrón clave (para memorizar)

```
Pregunta natural → Agente (razona) → Tool calling (ML, stats) → Síntesis en lenguaje natural
```